In [14]:
# Install dependencies
!pip install sentence-transformers faiss-cpu pypdf langchain langchain-text-splitters python-dotenv --quiet

import os
import re
import faiss
import torch
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

load_dotenv()

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"


In [34]:
#1 Document Ingestion
def load_pdf(path):
    """Load and extract text from a PDF file."""
    try:
        reader = PdfReader(path)
        pages = [p.extract_text() for p in reader.pages if p.extract_text()]
        if not pages:
            print(f"[Warning] PDF contains no extractable text: {path}")
        return "\n".join(pages)
    except Exception as e:
        print(f"[Error] Failed to read {path}: {e}")
        return ""

def load_text(path):
    """Load plain text or markdown file."""
    try:
        with open(path, "r", encoding="utf-8") as f:
            return f.read()
    except:
        with open(path, "r", encoding="latin-1") as f:
            return f.read()

def ingest_documents(folder):
    """Load PDF/TXT/MD files from a folder."""
    docs = {}
    for file in os.listdir(folder):
        path = os.path.join(folder, file)
        if file.endswith(".pdf"):
            text = load_pdf(path)
        elif file.endswith(".txt") or file.endswith(".md"):
            text = load_text(path)
        else:
            continue

        if len(text.strip()) > 0:
            docs[file] = text
    return docs


In [16]:
#2 Clean & Preprocess Documents
def clean_text(text):
    """Basic cleaning — whitespace, unicode fixes, dedupe."""
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def preprocess_documents(docs):
    """Apply cleaning to each loaded document."""
    return {name: clean_text(txt) for name, txt in docs.items()}


In [17]:
#3 Chunking / Splitting
def chunk_documents(docs, chunk_size=400, chunk_overlap=60):
    """Split documents into overlapping chunks for embeddings."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )

    chunks = []
    metadata = []

    for name, text in docs.items():
        doc_chunks = splitter.split_text(text)
        for idx, chunk in enumerate(doc_chunks):
            chunks.append(chunk)
            metadata.append({"source": name, "chunk_index": idx})

    return chunks, metadata


In [18]:
#4 Embed Chunks (Sentence Transformers)
def embed_chunks(chunks, model_name=EMBED_MODEL):
    model = SentenceTransformer(model_name)
    embeddings = model.encode(chunks, convert_to_numpy=True, show_progress_bar=True)
    return embeddings


In [19]:
#5 Build FAISS Index
def build_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings.astype("float32"))
    return index


In [20]:
#6 Retrieval Function
def retrieve(query, index, chunks, metadata, embed_model=EMBED_MODEL, k=4):
    model = SentenceTransformer(embed_model)
    q_emb = model.encode([query]).astype("float32")
    distances, idxs = index.search(q_emb, k)

    results = []
    for i in idxs[0]:
        results.append({
            "chunk": chunks[i],
            "metadata": metadata[i]
        })
    return results


In [22]:
!pip install transformers langchain-huggingface --quiet
#7 LLM Answer Generation
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

def load_local_llm():
    model_name = "facebook/opt-350m"

    tok = AutoTokenizer.from_pretrained(model_name)
    llm_model = AutoModelForCausalLM.from_pretrained(model_name)

    pipe = pipeline(
        "text-generation",
        model=llm_model,
        tokenizer=tok,
        max_new_tokens=150,
        do_sample=True
    )

    return HuggingFacePipeline(pipeline=pipe)

local_llm = load_local_llm()


tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/663M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/662M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Device set to use cpu


In [23]:
#8 Full Q&A Pipeline
def answer_query(query, index, chunks, metadata, use_openai=False):
    retrieved = retrieve(query, index, chunks, metadata)

    context = "\n\n".join([r["chunk"] for r in retrieved])

    if use_openai:
        return answer_with_openai(query, context)
    else:
        prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
        return local_llm(prompt)


In [29]:
#9 Test and Run
docs = ingest_documents("documents/")
docs = preprocess_documents(docs)
chunks, metadata = chunk_documents(docs)
embeddings = embed_chunks(chunks)
index = build_faiss_index(embeddings)

print("Pipeline Ready")


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Pipeline Ready


In [31]:
def answer_query(query, index, chunks, metadata, use_openai=False):

    #Retrieve chunks
    retrieved = retrieve(query, index, chunks, metadata)

    #Build context
    context = "\n\n".join([r["chunk"] for r in retrieved])

    #Build prompt
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"

    if use_openai:
        return answer_with_openai(query, context)
    else:
        # Correct API for LangChain LLMs
        response = local_llm.invoke(prompt)
        return response


In [32]:
answer_query("What is the key idea of the document?", index, chunks, metadata)


Both `max_new_tokens` (=150) and `max_length`(=21) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Context:\nand design system components or processes that meet the specified needs with appropriate consideration for the public health and safety, and the cultural, societal, and environmental considerations [PO.4]. Conduct investigations of complex problems : Use research -based knowledge and research methods including design of experiments, analysis and interpretation of data, and synthesis of the\n\nsettings [PO.10]. Communication: Communicate effectively on complex engineering activities with the engineering community and with society at large, such as, being able to comprehend and write effective reports and design documentation, make effective presentations, and give and receive clear instructions [PO.11]. Project management and finance: Demonstrate knowledge and understanding of the\n\nand searching) defined over it. Lecture CC0051 .1 CC0051 .2 Class Quiz Home Assignments I Sessional End Term 14. Circular Linked List: Introduction, Operations understand and implement circular l